In [1]:
import itertools

# x1-Темп, x2-Аппетит, x3-Активн, x4-Слизист, x5-Дыхание, x6-Шерсть, x7-ЖКТ
training_data = [
    # Здоровые
    {'x': (0, 0, 0, 0, 0, 0, 0), 'y': 1},
    {'x': (0, 0, 0, 0, 0, 1, 0), 'y': 1},
    {'x': (0, 0, 1, 0, 0, 0, 0), 'y': 1},
    {'x': (0, 0, 0, 1, 0, 0, 0), 'y': 1},
    {'x': (0, 0, 0, 0, 0, 0, 1), 'y': 1},
    {'x': (0, 1, 0, 0, 0, 0, 0), 'y': 1},
    {'x': (0, 0, 0, 0, 1, 0, 0), 'y': 1},
    # Больныe
    {'x': (1, 1, 1, 1, 1, 1, 1), 'y': 0},
    {'x': (1, 1, 1, 0, 0, 0, 0), 'y': 0},
    {'x': (1, 0, 1, 1, 1, 0, 0), 'y': 0},
    {'x': (0, 1, 1, 0, 0, 1, 1), 'y': 0},
    {'x': (1, 0, 0, 1, 0, 1, 1), 'y': 0},
    {'x': (0, 1, 0, 1, 1, 0, 1), 'y': 0},
    {'x': (1, 1, 0, 0, 1, 1, 0), 'y': 0},
]

def get_fragment(vector, indices):
    return tuple(vector[i] for i in indices)

def is_test(indices, data):
    objs_y1 = [get_fragment(d['x'], indices) for d in data if d['y'] == 1]
    objs_y2 = [get_fragment(d['x'], indices) for d in data if d['y'] == 0]
    for f1 in objs_y1:
        if f1 in objs_y2: return False
    return True

def find_deadlock_tests(data):
    n = len(data[0]['x'])
    deadlock_tests = []
    for r in range(1, n + 1):
        for subset in itertools.combinations(range(n), r):
            if is_test(subset, data):
                # Проверка на тупиковость (минимальность)
                if not any(set(dt).issubset(set(subset)) for dt in deadlock_tests):
                    deadlock_tests.append(subset)
        if deadlock_tests:
             if r >= 3: break
    return deadlock_tests

def calculate_votes(new_x, target_y, tests, data):
    class_objects = [d['x'] for d in data if d['y'] == target_y]
    if not tests: return 0
    total_score = 0
    for test in tests:
        new_f = get_fragment(new_x, test)
        for obj_x in class_objects:
            if new_f == get_fragment(obj_x, test):
                total_score += 1
    # Нормировка: сумма совпадений / (кол-во эталонов * кол-во тестов)
    return total_score / (len(class_objects) * len(tests))

tests = find_deadlock_tests(training_data)
print(f"Найдено тупиковых тестов: {len(tests)}")
print(f"Примеры тестов (индексы признаков): {tests[:5]}\n")

test_samples = [
    (0, 0, 0, 0, 0, 0, 0), # 1. Здоровое
    (1, 1, 1, 0, 0, 0, 1), # 2. Больное
    (0, 0, 1, 0, 0, 0, 0), # 3. Подозрение на здоровое
    (1, 0, 0, 1, 1, 0, 0), # 4. Больное
]

for i, obj in enumerate(test_samples, 1):
    g1 = calculate_votes(obj, 1, tests, training_data)
    g0 = calculate_votes(obj, 0, tests, training_data)
    result = "Здоровое (y1)" if g1 > g0 else "Больное (y2)"
    print(f"Тест №{i} {obj}: Г1={g1:.3f}, Г2={g0:.3f} -> Итог: {result}")

Найдено тупиковых тестов: 1
Примеры тестов (индексы признаков): [(0, 1, 6)]

Тест №1 (0, 0, 0, 0, 0, 0, 0): Г1=0.714, Г2=0.000 -> Итог: Здоровое (y1)
Тест №2 (1, 1, 1, 0, 0, 0, 1): Г1=0.000, Г2=0.143 -> Итог: Больное (y2)
Тест №3 (0, 0, 1, 0, 0, 0, 0): Г1=0.714, Г2=0.000 -> Итог: Здоровое (y1)
Тест №4 (1, 0, 0, 1, 1, 0, 0): Г1=0.000, Г2=0.143 -> Итог: Больное (y2)
